In [6]:
# # 3.02 — Baseline Regressão Logística
#
# Objetivo: treinar LogisticRegression como segundo baseline, comparar com
# o DummyClassifier e identificar as features mais relevantes via coeficientes.
#
# Decisões técnicas:
# - class_weight="balanced" para lidar com desbalanceamento residual no test set
# - solver="lbfgs" — eficiente para datasets de tamanho médio (~5k amostras)
# - max_iter=1000 — garante convergência
# - Validação cruzada estratificada (k=5) sobre X_train (já com SMOTE+ENN)
# - X_test intocado — avaliação final única, nunca usado no desenvolvimento

# %% Imports
import sys
from pathlib import Path

import mlflow
import mlflow.sklearn
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    average_precision_score,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import StratifiedKFold, cross_validate

# ── Adiciona raiz do projeto ao path ──────────────────────────────────────────
sys.path.insert(0, str(Path("..").resolve()))

from churn_telecom.config import (
    DATA_PROCESSED,
    RANDOM_STATE,
    REPORTS_FIGURES,
    get_logger,
    log_dataset_to_mlflow,
    mlflow_run,
    setup_mlflow,
)
from churn_telecom.plots import (
    save_confusion_matrix,
    save_feature_importance,
    save_roc_curve,
)

In [7]:
logger = get_logger("3.02_baseline_logistic")
logger.info("Iniciando script 3.02_vab_baseline_logistic.py")

22:36:08 | INFO | Iniciando script 3.02_vab_baseline_logistic.py


In [8]:
# ## 1. Carregamento dos dados
#
# Carrega train/test processados pelo pipeline do notebook 1.03.
# - train.parquet : 5395 amostras, 30 features (pós SMOTE+ENN, churn ~57%)
# - test.parquet  : 1405 amostras, 30 features (intocado,  churn ~26%)

# %% Carrega dados
TRAIN_PATH = DATA_PROCESSED / "train.parquet"
TEST_PATH = DATA_PROCESSED / "test.parquet"

assert TRAIN_PATH.exists(), f"Arquivo não encontrado: {TRAIN_PATH}"
assert TEST_PATH.exists(), f"Arquivo não encontrado: {TEST_PATH}"

train = pd.read_parquet(TRAIN_PATH)
test = pd.read_parquet(TEST_PATH)

TARGET = "churn_value"
X_train = train.drop(columns=[TARGET])
y_train = train[TARGET]
X_test = test.drop(columns=[TARGET])
y_test = test[TARGET]

feature_names: list[str] = X_train.columns.tolist()

logger.info("Train : %s | churn: %.2f%%", X_train.shape, y_train.mean() * 100)
logger.info("Test  : %s | churn: %.2f%%", X_test.shape, y_test.mean() * 100)
logger.info("Features: %d", len(feature_names))

22:36:08 | INFO | Train : (5395, 30) | churn: 57.41%
22:36:08 | INFO | Test  : (1405, 30) | churn: 26.48%
22:36:08 | INFO | Features: 30


In [9]:
# ## 2. Treinamento e registro no MLflow
#
# Fluxo dentro do run:
# 1. Tags descritivas (notebook, fase, modelo, framework)
# 2. Dataset versionado via hash MD5 (train + test)
# 3. Parâmetros do modelo
# 4. Validação cruzada estratificada (k=5)
# 5. Treino final no X_train completo
# 6. Predições no X_test (uso único)
# 7. Métricas: AUC-ROC, PR-AUC, F1, Recall, Precision + desvios CV
# 8. Artefatos: confusion matrix, ROC curve, feature importance
# 9. Registro no Model Registry

# %% Treino e registro
setup_mlflow()

MODEL_NAME = "baseline-logistic"  # nome no Model Registry — sem barras nem pontos

with mlflow_run("3.02_baseline_logistic") as run:
    # ── 1. Tags ────────────────────────────────────────────────────────────────
    mlflow.set_tags(
        {
            "notebook": "3.02",
            "fase": "baseline",
            "modelo": "logistic",
            "task": "classification",
            "framework": "sklearn",
        }
    )

    # ── 2. Dataset versionado ─────────────────────────────────────────────────
    log_dataset_to_mlflow(X_train, y_train, split="train", source_path=TRAIN_PATH)
    log_dataset_to_mlflow(X_test, y_test, split="test", source_path=TEST_PATH)

    # ── 3. Parâmetros ─────────────────────────────────────────────────────────
    params = {
        "class_weight": "balanced",  # compensa desbalanceamento no test set
        "max_iter": 1000,  # garante convergência
        "random_state": RANDOM_STATE,
        "solver": "lbfgs",
        "C": 1.0,  # regularização L2 padrão
        "cv_folds": 5,
    }
    mlflow.log_params(params)
    logger.info("Params: %s", params)

    # ── 4. Validação cruzada estratificada ────────────────────────────────────
    model = LogisticRegression(
        class_weight=params["class_weight"],
        max_iter=params["max_iter"],
        random_state=params["random_state"],
        solver=params["solver"],
        C=params["C"],
    )
    cv = StratifiedKFold(
        n_splits=params["cv_folds"],
        shuffle=True,
        random_state=RANDOM_STATE,
    )
    cv_results = cross_validate(
        model,
        X_train,
        y_train,
        cv=cv,
        scoring=["average_precision", "roc_auc", "f1", "recall", "precision"],
    )
    logger.info(
        "CV AUC  : %.4f ± %.4f",
        cv_results["test_roc_auc"].mean(),
        cv_results["test_roc_auc"].std(),
    )
    logger.info(
        "CV F1   : %.4f ± %.4f",
        cv_results["test_f1"].mean(),
        cv_results["test_f1"].std(),
    )
    logger.info(
        "CV Recall: %.4f ± %.4f",
        cv_results["test_recall"].mean(),
        cv_results["test_recall"].std(),
    )

    # ── 5. Treino final + predições no test set ───────────────────────────────
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    y_proba = model.predict_proba(X_test)[:, 1]

    # ── 6. Métricas ───────────────────────────────────────────────────────────
    metrics = {
        # Test set — avaliação final
        "test_auc": roc_auc_score(y_test, y_proba),
        "test_pr_auc": average_precision_score(y_test, y_proba),
        "test_f1": f1_score(y_test, y_pred),
        "test_recall": recall_score(y_test, y_pred),
        "test_precision": precision_score(y_test, y_pred),
        # Cross-val — estabilidade do modelo
        "cv_auc_mean": float(cv_results["test_roc_auc"].mean()),
        "cv_auc_std": float(cv_results["test_roc_auc"].std()),
        "cv_pr_auc_mean": float(cv_results["test_average_precision"].mean()),
        "cv_pr_auc_std": float(cv_results["test_average_precision"].std()),
        "cv_f1_mean": float(cv_results["test_f1"].mean()),
        "cv_f1_std": float(cv_results["test_f1"].std()),
        "cv_recall_mean": float(cv_results["test_recall"].mean()),
        "cv_recall_std": float(cv_results["test_recall"].std()),
    }
    mlflow.log_metrics(metrics)
    logger.info("Metrics: %s", {k: f"{v:.4f}" for k, v in metrics.items()})

    # ── 7. Artefatos visuais ──────────────────────────────────────────────────
    base = REPORTS_FIGURES / "baselines"

    # Confusion Matrix
    mlflow.log_artifact(
        str(
            save_confusion_matrix(
                y_test,
                y_pred,
                base / "logistic_confusion_matrix.png",
                title="Confusion Matrix — Logistic Regression",
            )
        ),
        artifact_path="baselines/logistic",
    )

    # ROC Curve
    mlflow.log_artifact(
        str(
            save_roc_curve(
                y_test,
                y_proba,
                base / "logistic_roc_curve.png",
                model_name="Logistic Regression",
                title="ROC Curve — Logistic Regression",
            )
        ),
        artifact_path="baselines/logistic",
    )

    # Feature Importance via |coeficientes|
    # coef_ shape: (1, n_features) → ravel() para 1D
    coef_abs = np.abs(model.coef_.ravel())
    mlflow.log_artifact(
        str(
            save_feature_importance(
                importances=coef_abs,
                feature_names=feature_names,
                path=base / "logistic_feature_importance.png",
                title="Feature Importance — |Coeficientes| Logistic Regression",
                color="steelblue",
                top_n=20,
            )
        ),
        artifact_path="baselines/logistic",
    )

    # ── Loga coeficientes como dict para análise posterior ────────────────────
    coef_dict = {
        name: float(round(coef, 6))
        for name, coef in zip(feature_names, model.coef_.ravel())
    }
    mlflow.log_dict(coef_dict, "baselines/logistic/coefficients.json")
    logger.info("Top 5 features por |coef|:")
    top5 = sorted(coef_dict.items(), key=lambda x: abs(x[1]), reverse=True)[:5]
    for feat, val in top5:
        logger.info("  %-35s : %+.4f", feat, val)

    # ── 8. Analisa gender e phone_service ────────────────────────────────────
    # Verifica importância relativa para decidir descarte na Etapa 2
    gender_phone_cols = [
        c for c in feature_names if "gender" in c or "phone_service" in c
    ]
    if gender_phone_cols:
        gp_importance = {c: float(abs(coef_dict[c])) for c in gender_phone_cols}
        mlflow.log_dict(
            gp_importance, "baselines/logistic/gender_phone_importance.json"
        )
        logger.info("Importância gender/phone_service: %s", gp_importance)
    else:
        logger.info("Colunas gender/phone_service não encontradas nas features.")

    # ── 9. Log + registro no Model Registry ──────────────────────────────────
    model_info = mlflow.sklearn.log_model(
        sk_model=model,
        artifact_path="model",  # sem barras — MLflow >= 2.18
        registered_model_name=MODEL_NAME,  # cria Version 1 no Registry
        input_example=X_test.iloc[:5],
    )

    logger.info("Model URI : %s", model_info.model_uri)
    logger.info("Run ID    : %s", run.info.run_id)

c:\Users\victo\Desktop\MLOPS\AULAS\tech_challenge_fase_1\projeto_final\.venv\lib\site-packages\mlflow\data\dataset_source_registry.py:148: UserWarning: The specified dataset source can be interpreted in multiple ways: LocalArtifactDatasetSource, LocalArtifactDatasetSource. MLflow will assume that this is a LocalArtifactDatasetSource source.
  return _dataset_source_registry.resolve(
c:\Users\victo\Desktop\MLOPS\AULAS\tech_challenge_fase_1\projeto_final\.venv\lib\site-packages\mlflow\types\utils.py:440: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns

Registered model 'baseline-logistic' already exists. Creating a new version of this model...
Created version '2' of model 'baseline-logistic'.
22:36:12 | INFO | Model URI : models:/m-725d6a079be74a7bbacd83d137fdf9b7
22:36:12 | INFO | Run ID    : ce0f6332e12c4e908c34c1f0442ee3fa


In [10]:
# ## 3. Análise rápida dos coeficientes
#
# Células abaixo são exploratórias — não afetam o registro no MLflow.

# %% Exibe top 10 features por magnitude do coeficiente
coef_series = pd.Series(model.coef_.ravel(), index=feature_names).reindex(
    pd.Series(model.coef_.ravel(), index=feature_names)
    .abs()
    .sort_values(ascending=False)
    .index
)

logger.info("\nTop 10 coeficientes (magnitude):\n%s", coef_series.head(10).to_string())

22:36:12 | INFO | 
Top 10 coeficientes (magnitude):
dependents            -2.697160
total_charges         -1.579788
internet_service_No   -1.538883
charges_per_month      1.475288
phone_service         -1.337249
contract_Two year     -1.314595
is_month_to_month      1.243296
tenure_months          1.080626
paperless_billing      0.916891
is_fiber_optic         0.911392
